In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import importlib
import json
import os
import sys

import boto3
import pandas as pd

MODEL_ID = "usecase2"

# Reference inputs from a completed sample run. For your failure, replace all
# three URIs with the failed Preprocess job's actual inputs and source parameters.
# Cached inputs can belong to an earlier execution.
S3_INPUTS = {
    "data": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/data/raw/data.csv",
    "selection": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/data/selection/selected_features.csv",
    "parameters": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/reports/evaluation/parameters.json",
}

# The checkout supplies the code being edited; S3 supplies the input files.
REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
sys.path.insert(0, str(REPOSITORY / "mlops-models/src"))

work_directory = Path(mkdtemp(prefix="preprocess-training-"))
data_directory = work_directory / "data"
selection_file = work_directory / "selection" / "selected_features.csv"
parameter_path = work_directory / "settings" / MODEL_ID / "parameters.json"
output_directory = work_directory / "output"
output_directory.mkdir()

os.environ.update({
    "MODEL_ID": MODEL_ID,
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
})

# Download the listed files unchanged. Nothing is uploaded or deployed.
s3 = boto3.client("s3", region_name="us-east-1")
local_files = {
    "data": data_directory / "data.csv",
    "selection": selection_file,
    "parameters": parameter_path,
}
for name, local_file in local_files.items():
    local_file.parent.mkdir(parents=True, exist_ok=True)
    address = urlsplit(S3_INPUTS[name])
    s3.download_file(address.netloc, address.path.lstrip("/"), str(local_file))

parameters = json.loads(parameter_path.read_text())


In [ ]:
# Inspect the inputs before running preprocessing.
print("Code checkout:", REPOSITORY)
print("Input files (S3 -> local):")
for name, local_file in local_files.items():
    print(f"  {name}: {S3_INPUTS[name]} -> {local_file}")
print("Local output:", output_directory / "prepared" / "data.csv")
display(parameters)

data_columns = pd.read_csv(data_directory / "data.csv", nrows=0).columns.tolist()
selection = pd.read_csv(selection_file)
selected_columns = selection.iloc[:, 0].tolist()
primary_key = parameters.get("primary_key")
required_columns = list(dict.fromkeys(
    ([primary_key] if primary_key is not None else []) + selected_columns
))

print("Input columns:", data_columns)
print("Missing parameter keys:", [key for key in ("primary_key",) if key not in parameters])
print("Missing identifier/selected columns:",
      [column for column in required_columns if column not in data_columns])
display(selection)
display(pd.read_csv(data_directory / "data.csv", nrows=10, dtype=str))


In [ ]:
from mlops_models.model_base import preprocess

# Use this checkout, not an older module retained from another project.
expected_source = REPOSITORY / "mlops-models/src/mlops_models/model_base/preprocess.py"
assert Path(preprocess.__file__).resolve() == expected_source.resolve(), (
    f"Loaded {preprocess.__file__}. Restart the kernel in {REPOSITORY}."
)
preprocess = importlib.reload(preprocess)
print("Preprocessing source:", Path(preprocess.__file__).resolve())

# This calls the base function directly, as in the other step notebooks.
# Base preprocessing reads parameters.json; it does not use training metadata.
# Rerun this cell after editing preprocess.py. Exceptions remain visible here.
result_directory = preprocess.process_train(
    data_directory=data_directory,
    selection_file=selection_file,
    output_directory=output_directory,
    config={},
    parameter_path=parameter_path,
)
print("Prepared CSV:", result_directory / "data.csv")
display(pd.read_csv(result_directory / "data.csv", nrows=10, dtype=str))
result_directory
